In [2]:
# =============================================================================
# Compare Original vs Augmented (excluding fallback rows)
# Run this in a Jupyter notebook or .ipynb file
# =============================================================================

import pandas as pd
from IPython.display import display, HTML
import random

# ── Paths (update these) ────────────────────────────────────────────
ORIGINAL_REAL  = r"../aug/real_120k_matched.csv"
ORIGINAL_FAKE  = r"../aug/fake_120k_matched.csv"
AUGMENTED_REAL = r"../aug/real_120k_t5paws_aug.csv"
AUGMENTED_FAKE = r"../aug/fake_120k_t5paws_aug.csv"

SAMPLE_SIZE = 10          # how many random examples to show
SEED = 42                 # reproducible random sample

# ── Load and merge real + fake ──────────────────────────────────────
print("Loading data...")

df_orig_real = pd.read_csv(ORIGINAL_REAL, usecols=["text", "label"])
df_orig_fake = pd.read_csv(ORIGINAL_FAKE, usecols=["text", "label"])

df_aug_real = pd.read_csv(AUGMENTED_REAL)
df_aug_fake = pd.read_csv(AUGMENTED_FAKE)

# Add source column
df_orig_real["source"] = "real_original"
df_orig_fake["source"] = "fake_original"
df_aug_real["source"] = "real_aug"
df_aug_fake["source"] = "fake_aug"

# Merge original + augmented (match by index)
df_real = pd.concat([df_orig_real, df_aug_real], axis=1, keys=["orig", "aug"])
df_fake = pd.concat([df_orig_fake, df_aug_fake], axis=1, keys=["orig", "aug"])

# Combine real & fake
df_combined = pd.concat([df_real, df_fake], ignore_index=True)

# Filter only successful paraphrases (no fallback)
df_success = df_combined[df_combined[("aug", "aug_type")] != "fallback"].copy()

print(f"Total successful paraphrases: {len(df_success):,} "
      f"({len(df_success)/len(df_combined)*100:.2f}% of total)")

# ── Add word count difference ───────────────────────────────────────
df_success[("diff", "word_diff")] = (
    df_success[("aug", "text")].str.split().str.len() -
    df_success[("orig", "text")].str.split().str.len()
)

# ── Sample random examples ──────────────────────────────────────────
random.seed(SEED)
sample_df = df_success.sample(n=min(SAMPLE_SIZE, len(df_success)))

# ── Pretty display function ─────────────────────────────────────────
def show_side_by_side(df_sample):
    html = """
    <style>
        table { width:100%; border-collapse: collapse; font-family: Arial, sans-serif; }
        th, td { padding: 10px; text-align: left; vertical-align: top; border: 1px solid #ccc; }
        th { background-color: #f2f2f2; font-weight: bold; }
        .row-even { background-color: #f9f9f9; }
        .row-odd { background-color: #ffffff; }
        .diff-positive { color: green; font-weight: bold; }
        .diff-negative { color: red; font-weight: bold; }
    </style>
    <table>
        <tr>
            <th>#</th>
            <th>Original Text (up to 800 chars)</th>
            <th>Augmented Text (up to 800 chars)</th>
            <th>Word Diff</th>
            <th>Label / Source</th>
        </tr>
    """

    for idx, (i, row) in enumerate(df_sample.iterrows()):
        orig = str(row[("orig", "text")])[:800] + ("..." if len(str(row[("orig", "text")])) > 800 else "")
        aug  = str(row[("aug", "text")])[:800]  + ("..." if len(str(row[("aug", "text")]))  > 800 else "")
        diff = row[("diff", "word_diff")]
        label_source = f"Label: {row[('orig', 'label')]} | {row[('aug', 'source')]}"

        diff_class = "diff-positive" if diff > 0 else "diff-negative" if diff < 0 else ""
        row_class = "row-even" if idx % 2 == 0 else "row-odd"

        html += f"""
        <tr class="{row_class}">
            <td>{i}</td>
            <td>{orig.replace('\n', '<br>')}</td>
            <td>{aug.replace('\n', '<br>')}</td>
            <td class="{diff_class}">{diff:+d}</td>
            <td>{label_source}</td>
        </tr>
        """

    html += "</table>"
    display(HTML(html))

# Then call it again:

# ── Show results ─────────────────────────────────────────────────────
print(f"\nRandom sample of {len(sample_df)} successful paraphrases:\n")
show_side_by_side(sample_df)

# Optional: summary stats
print("\nWord difference statistics (successful paraphrases only):")
print(df_success[("diff", "word_diff")].describe().round(2))

Loading data...
Total successful paraphrases: 239,892 (99.95% of total)

Random sample of 10 successful paraphrases:



#,Original Text (up to 800 chars),Augmented Text (up to 800 chars),Word Diff,Label / Source
173418,"“Reporters, including CNN’s Cooper, beaten in Egypt” “Reporters, including CNN’s Cooper, beaten in Egypt” (Before It's News) Of course, not by the peaceful protesters just yearning for self-government and individual liberty. No. Because we all know the only thuggery exists on the side of the government who has insisted for years that it’s just not right to try to eliminate the Jews. Theater. In a way, Cooper got lucky. Had the little swish visited Egyptonce the Muslim Brotherhood takes power he likely would have been buried up to his chest in the sand and then stoned to death. For freedom. Read more at Protein Wisdom Source:","In a way, Cooper got lucky. Had the little swish visited Egypt once the Muslim Brotherhood takes power he likely would have been buried in the sand and then stoned to death for freedom .",-71,Label: 1 | fake_aug
161052,"SHILLING FOR THE MET OFFICE... Headline: Bitcoin & Blockchain Searches Exceed Trump! Blockchain Stocks Are Next! Seen this from the BBC? “British winters are likely to become milder and wetter like the last one but cold spells still need to be planned for, says the UK Met Office. Summers are likely to be hotter and drier, but washouts are still on the cards, it adds.”","Bitcoin & Blockchain Searches Exceed Trump! Blockchain Stocks Are Next! Seen this from the BBC? “British winters are likely to become milder and wetter like the last, but cold spells still need to be planned for, says the UK Met Office. Summers are likely to be hotter and drier, but washouts are still on the cards, it adds .",-6,Label: 1 | fake_aug
219037,"Rand Paul Leads Among Republicans in New Hampshire Rand Paul Leads Among Republicans in New Hampshire (Before It's News) There is still a long way to go before the 2016 presidential election, but Public Policy Polling has a new survey of New Hampshire that gives Sen. Rand Paul (R-KY) some very early bragging rights. According to the survey, the Paul leads Sen. Marco Rubio (R-FL), Gov. Chris Christie (R-NJ), and the rest of the field in what has been a tone-setting state: PPP’s new poll of New Hampshire Republicans about 2016 finds momentum on Rand Paul’s side. He leads the potential field with 28% to 25% for Marco Rubio, 14% for Chris Christie, 7% for Jeb Bush and Paul Ryan, 4% for Rick Santorum, 3% for Susana Martinez, and 1% each for Rick Perry and Bobby Jindal. Paul has seen a huge incr...","There is still a long way to go before the 2016 presidential election, but Public Policy Polling has a new poll of New Hampshire Republicans that finds momentum on Rand Paul (R-KY) . He leads the potential field with 28% to 25% for Marco Rubio (R-FL), 14% for Chris Christie (R-NJ) and 7% for Jeb Bush and Paul Ryan, 4% for Rick Santorum, 3% for Susana Martinez and 1% each for Rick Perry and Bobby Jindal .",-244,Label: 1 | fake_aug
14170,"Recipe: Salmon Sandwich Extraordinaire Let's Get Started Prior to receiving The New Essentials of French Cooking for free, please confirm your email address below. Prior to your purchase of The New Essentials of French Cooking for $1.99, please confirm your email address below. Prior to your purchase of The New Essentials of French Cooking for $4.99, please confirm your email address below. Prior to your purchase of The New Essentials of French Cooking for $9.99, please confirm your email address below.","Recipe: Salmon Sandwich Extraordinaire Let's Get Started Prior to receiving The New Essentials of French Cooking for free, please confirm your email address below . Prior to your purchase of The New Essentials of French Cooking for $1.99 , please confirm your email address below . Prior to your purchase of The New Essentials of French Cooking for $4.99 , please confirm your email address below .",-14,Label: 0 | real_aug
159117,"The Unwinding of the Social Security Ponzi Scheme The Unwinding of the Social Security Ponzi Scheme % of readers think this


Word difference statistics (successful paraphrases only):
count    239892.00
mean       -142.06
std         123.97
min        -493.00
25%        -233.00
50%        -112.00
75%         -30.00
max          25.00
Name: (diff, word_diff), dtype: float64
